# Adding Metadata

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader 
from pathlib import Path

path_to_files = Path("../data/interim")

list_of_documents = os.listdir(path_to_files)
list_of_documents

all_docs = []

def get_metadata_from_file_name(file_name: str):

    parts = file_name.replace(".pdf", "").split("_")

    return {
        "niveau": parts[1] if len(parts) > 1 else "unknown",
        "classe": parts[2] if len(parts) > 2 else "unknown",
        "voie": parts[3] if len(parts) > 3 else "unknown",
        "filiere": parts[4] if len(parts) > 4 else "unknown",
        "annee": parts[-1] if parts[-1].isdigit() else "unknown"
    }

all_docs = []

for file in list_of_documents:

    path = Path(path_to_files, file)
    loader = PyMuPDFLoader(str(path))
    docs = loader.load()

    # 🔥 1. metadata calculée UNE seule fois
    file_meta = get_metadata_from_file_name(file)
    url =  "https://eduscol.education.gouv.fr/5817/programmes-et-ressources-en-mathematiques-voie-gt"


    global_meta = {
        **file_meta,
        "source": url,
        "file_name": file,
        "title": file.replace(".pdf", "")
    }

    # 🔥 2. propagation à toutes les pages
    for d in docs:

        d.metadata = {
            **d.metadata,   # garde page + infos loader
            **global_meta   # ajoute metadata PDF
        }

    # 🔥 3. ajout au corpus final
    all_docs.extend(docs) 

all_docs[0].metadata

{'producer': 'Microsoft® Word 2010',
 'creator': 'Microsoft® Word 2010',
 'creationdate': '2019-01-18T15:43:47+01:00',
 'source': 'https://eduscol.education.gouv.fr/5817/programmes-et-ressources-en-mathematiques-voie-gt',
 'file_path': '../data/interim/BO_lycee_seconde_sthr_standard_2019.pdf',
 'total_pages': 13,
 'format': 'PDF 1.5',
 'title': 'BO_lycee_seconde_sthr_standard_2019',
 'author': 'Utilisateur',
 'subject': '',
 'keywords': '',
 'moddate': '2019-01-18T15:57:18+01:00',
 'trapped': '',
 'modDate': "D:20190118155718+01'00'",
 'creationDate': "D:20190118154347+01'00'",
 'page': 0,
 'niveau': 'lycee',
 'classe': 'seconde',
 'voie': 'sthr',
 'filiere': 'standard',
 'annee': '2019',
 'file_name': 'BO_lycee_seconde_sthr_standard_2019.pdf'}

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os

# remonter à la racine du projet
env_path = Path("..") / ".env"

load_dotenv(env_path)

print(os.getenv("MISTRAL_API_KEY"))

osZDGy23YzF4yFZAlKw6OCF03NZcnDKt


In [ ]:
"""from langchain_experimental.text_splitter import SemanticChunker

splitter = SemanticChunker(
    embeddings,
)

chunks = splitter.create_documents(
    [doc.page_content for doc in all_docs],
    metadatas=[doc.metadata for doc in all_docs]
)""""

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunker = RecursiveCharacterTextSplitter(
        chunk_size=4000, # à changer  
        chunk_overlap=50, # à changer 
        separators=["\n\n","\n","."," ",""]
    )
# split document prends une liste de l'object documents 
chunks = chunker.split_documents(documents=all_docs)


In [4]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings()

# Step 3: Embeddings and storing  
# -------------------------------
vector_store = InMemoryVectorStore(embedding=MistralAIEmbeddings(
    model="mistral-embed",
))
vector_store.add_documents(chunks)

['73d62d65-e48d-4786-a61c-52e080b9684c',
 'da74c167-7b75-490d-9c5f-9d7cef4658bb',
 'a802d40f-1ff0-42e6-997b-a8d2f46f5bfb',
 '20bdbdce-cc18-4eaf-bc4d-7d7218d3a023',
 '79af5c08-a0e0-49c7-a74e-829c8806ae14',
 '4dbd4553-98a8-4bbd-846b-8c4fc1e6b984',
 '6148bf30-9303-4ed8-b0d8-ebaa459086f5',
 'f8995f27-e2a7-4a29-aff7-c643f33a384f',
 '2dd61d9a-1345-4c90-92e3-52597c6da1fe',
 'ebd0f2bb-d99a-4942-930b-3806dd7d736e',
 '9e303c63-fff4-46a0-8d55-526f79a34023',
 '587ad830-25bb-473d-8b94-22cd419c02b2',
 'f8268a45-6227-4e0c-8181-bee6f4358d82',
 '066cdc10-5d52-41a1-9abb-9a6b7056825a',
 '2b17a03c-b521-496c-b1ee-b4b9866ae88b',
 '86caac2e-7ddc-447a-88fc-133d9e0b0d3d',
 '3eeac05e-97bf-483f-a704-bd54bd0ab576',
 '784ea041-6530-4854-99cb-a72f3ff0341c',
 'cb6365ca-bc8a-4ed0-91c0-085346a3fbaa',
 '0a4f6164-de8f-4771-82d6-5d44d616ce8b',
 '11c5ac5c-8aeb-42b5-a87f-ad3168aefaed',
 'e878e0f6-6c92-4063-9a03-6825b405b705',
 '3a691bc8-b723-4f5d-85e6-7e4e6441cc67',
 '7a8ec4a4-e0b5-49fb-a250-58ddd08f604c',
 'e00737b1-fd41-

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_classic.retrievers import EnsembleRetriever, BM25Retriever
from langchain_core.prompts import ChatPromptTemplate

# =========================================================
# 1. LLM
# =========================================================
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2
)

# =========================================================
# 2. PROMPT ULTRA STRICT
# =========================================================
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant chargé d'extraire des informations.

RÈGLES ABSOLUES :

- Tu dois répondre UNIQUEMENT à partir du contexte fourni.
- Tu dois COPIER EXACTEMENT les phrases du contexte (aucune reformulation).
- Tu peux concaténer PLUSIEURS passages s'ils sont pertinents et complémentaires.
- Tu dois restituer la liste COMPLETE si elle est présente sur plusieurs parties du contexte.

INTERDICTIONS STRICTES :

- Ne reformule jamais
- N'explique jamais
- N'ajoute aucun mot
- Ne résume pas
- Ne complète pas

FORMAT DE RÉPONSE :

- Copier-coller EXACT des passages du contexte
- Conserver les tirets, retours à la ligne et ponctuation
- Respecter STRICTEMENT le texte original
- Garder l'ordre logique du document

Si aucune réponse n'est trouvée :
"je ne sais pas"

Contexte :
{context}

Question :
{question}

Réponse :
""")

# ==========================================================
# Query
# ==========================================================
question = """ Quelle est l'histoire des mathématiques sur la géométrie
            """


# ==========================================================
# 4. Rietreval Hybride
# ==========================================================

# 1. BM25
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 20

# 2. Vector retriever
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 20}
)

print(vector_retriever)
# 3. Hybrid
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.7, 0.3]
)

# 4. Retrieval HYBRIDE (IMPORTANT)
docs = hybrid_retriever.invoke(question)

# ==========================================================
# 🔥 FILTRAGE MÉTIER (CRITIQUE)
# ==========================================================

filtered_docs = [
    d for d in docs
    if d.metadata.get("classe") == "premiere"
    and d.metadata.get("voie") == "generale"
    and d.metadata.get("annee") == "2026"
]

# ==========================================================
# DEBUG (très important)
# ==========================================================

print("\n===== DOCS FILTRÉS =====\n")
for d in filtered_docs:
    print(d)
    print(f'******* classe : {d.metadata.get("classe","")}')
    print(f'******* voie : {d.metadata.get("voie","")}')
    print(f'******* année : {d.metadata.get("annee","")}')
    print(f"****** titre : {d.metadata.get('title','')}")
    print("------")

# ==========================================================
# Reranking 
# ==========================================================
def rerank_documents(query, docs, top_k=5):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)
    
    scored_docs = list(zip(docs, scores))
        
    # tri décroissant
    ranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    
    return [doc for doc, _ in ranked_docs[:top_k]]


reranked_docs = rerank_documents(question, filtered_docs, top_k=5)

# ==========================================================
# CONTEXT FINAL
# ==========================================================

context = "\n\n".join([doc.page_content for doc in reranked_docs])

# ==========================================================
# CHAIN
# ==========================================================

chain = prompt | llm

response = chain.invoke({
    "context": context,
    "question": question
})

print("réponse LLM -----------------")
print(response.content)

tags=['InMemoryVectorStore', 'MistralAIEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x7b70468048c0> search_kwargs={'k': 20}

===== DOCS FILTRÉS =====

page_content='Géométrie  
Objectifs  
L’étude de la géométrie plane menée au collège et en seconde a familiarisé les élèves à la géométrie de configuration, au calcul 
vectoriel et à la géométrie repérée. En première, on poursuit l’étude de la géométrie plane en introduisant de nouveaux outils. 
L’enseignement est organisé autour des objectifs suivants :  
− donner de nouveaux outils efficaces en vue de la résolution de problèmes géométriques, du point de vue métrique 
(produit scalaire) ;  
− enrichir la géométrie repérée de manière à pouvoir traiter des problèmes faisant intervenir l’orthogonalité.  
Les élèves doivent conserver une pratique du calcul vectoriel en géométrie non repérée.  
Histoire des mathématiques  
La notion de vecteur était implicite en mécanique depuis Galilée mais a

In [8]:
# 1. BM25
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 3

# 2. Vector retriever
vector_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print(vector_retriever)
# 3. Hybrid
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.7, 0.3]
)

# 4. Retrieval HYBRIDE (IMPORTANT)
docs = hybrid_retriever.invoke(question)

# 5. Context
context = "\n\n".join([doc.page_content for doc in docs])

# 6. LLM chain
chain = prompt | llm

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

tags=['InMemoryVectorStore', 'MistralAIEmbeddings'] vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x7b70468048c0> search_kwargs={'k': 3}
- Démonstrations
  − Calcul du terme général d’une suite arithmétique, d’une suite géométrique.
  − Calcul de 1 + 2 + … + 𝑛.
  − Calcul de 1 + 𝑞 + … + 𝑞𝑛.
